# PPS-SOHO Phase A — CIFAR-100 train-only gate
This notebook tests viability on frozen ViT features. It physically hides `test.pt` before selection and does **not** report held-out accuracy. Run cells in order.

In [ ]:
# === Edit this cell only ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/pps-soho'  # this branch must be pushed first
CHECKPOINT_SOURCE = 'huggingface'  # or 'google_drive'
DRIVE_CHECKPOINT_PATH = '/content/drive/MyDrive/T-SOHO/model.safetensors'
WORK_DIR = '/content/SOHO-CL'
CACHE_DIR = '/content/tsoho_cifar100_cache'
OUTPUT_DIR = '/content/pps_soho_phasea_outputs'
SEED = 1993
NUM_TASKS = 10
BATCH_SIZE = 128
CHECKPOINT_SIZE = 346284714
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'

In [ ]:
# Clean clone from a valid current directory.
import os, shutil, subprocess, sys, json, time
from pathlib import Path
os.chdir('/content')
shutil.rmtree(WORK_DIR, ignore_errors=True)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_GIT_URL, WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt', 'kagglehub', 'huggingface_hub'], check=True)
if CHECKPOINT_SOURCE == 'google_drive':
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_PATH = DRIVE_CHECKPOINT_PATH
elif CHECKPOINT_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
else:
    raise ValueError("CHECKPOINT_SOURCE must be 'huggingface' or 'google_drive'")
print('repo commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('checkpoint:', CHECKPOINT_PATH)
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
# Download CIFAR-100 only and verify the real backbone checkpoint.
import kagglehub
downloaded = Path(kagglehub.dataset_download('zaphat206/cifar-100'))
candidates = [downloaded, *downloaded.rglob('cifar-100')]
cifar_dir = next(p for p in candidates if (p/'train').is_file() and (p/'test').is_file() and (p/'meta').is_file())
CIFAR_ROOT = str(cifar_dir)
print('CIFAR-100:', CIFAR_ROOT)
subprocess.run([sys.executable, 'tools/checkpoint_preflight.py', '--root', CIFAR_ROOT, '--checkpoint', CHECKPOINT_PATH, '--checkpoint-size', str(CHECKPOINT_SIZE), '--checkpoint-sha256', CHECKPOINT_SHA256, '--seed', str(SEED), '--batch-size', str(BATCH_SIZE)], check=True)

In [ ]:
# Local correctness gate. Expected: all tests pass; sparse-CSC beta warning is non-fatal.
command = [sys.executable, '-m', 'pytest', '-q', 'tests/test_pps_soho_math.py', 'tests/test_pps_soho_learner.py', 'tests/test_experiment_runner.py']
print('Running:', ' '.join(command), flush=True)
subprocess.run(command, check=True)

In [ ]:
# Restore the frozen-feature cache if present; otherwise extract it once with live task progress.
metadata_path = Path(CACHE_DIR) / 'metadata.json'
if metadata_path.is_file():
    print('Using existing frozen-feature cache:', CACHE_DIR)
else:
    command = [sys.executable, '-u', 'tools/experiment_runner.py', '--extract-features-only', '--root', CIFAR_ROOT, '--backbone-checkpoint', CHECKPOINT_PATH, '--backbone-checkpoint-size', str(CHECKPOINT_SIZE), '--backbone-checkpoint-sha256', CHECKPOINT_SHA256, '--feature-cache-dir', CACHE_DIR, '--output-dir', f'{OUTPUT_DIR}/cache_extract', '--dataset', 'CIFAR-100', '--model-name', 'vit_base_patch16_224', '--data-augmentation', 'vit', '--seed', str(SEED), '--num-classes', '100', '--num-tasks', str(NUM_TASKS), '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--num-workers', '2']
    print('Extracting: each train/test task prints one progress line.', flush=True)
    subprocess.run(command, check=True)
metadata = json.loads(metadata_path.read_text())
print(json.dumps(metadata, indent=2))
assert metadata['feature_dim'] == 768 and metadata['finite'] is True

In [ ]:
# Physically lock the held-out cache, then run 16 train-only candidates with live START/DONE lines.
test_path = Path(CACHE_DIR) / 'test.pt'
locked_test_path = Path(CACHE_DIR) / 'test.locked.pt'
if test_path.is_file():
    test_path.replace(locked_test_path)
assert not test_path.exists(), 'test.pt must be hidden during selection'
selection_path = f'{OUTPUT_DIR}/selection.json'
command = [sys.executable, '-u', 'tools/experiment_runner.py', '--select-config', '--config', 'configs/pps_soho_cifar100_pilot.json', '--feature-cache-dir', CACHE_DIR, '--output-dir', f'{OUTPUT_DIR}/selection', '--selection-output', selection_path, '--device', 'cuda']
print('Starting PPS-SOHO train-only pilot. Wait for each DONE line.', flush=True)
started = time.time()
subprocess.run(command, check=True)
print(f'Selection complete in {(time.time()-started)/60:.1f} minutes')

In [ ]:
# Compact train-only report and preregistered gate. This cell does not restore or open test.pt.
import pandas as pd
payload = json.loads(Path(selection_path).read_text())
table = pd.DataFrame(payload['candidates'])
best = table.sort_values('validation_average_accuracy', ascending=False).groupby('method', as_index=False).first()
display(best[['method','rank','ridge_lambda','pps_gamma','validation_average_accuracy','persistent_state_bytes','solver_relative_residual_max','candidate_seconds']])
row = {item['method']: item for item in best.to_dict('records')}
protected = row['pps_class_protected']; standard = row['pps_standard_fd']; fly = row['cached_flycl']
gates = {
    'numerical_stability': max(v for v in table[table.method.str.startswith('pps_')].solver_relative_residual_max.dropna()) <= 1e-4,
    'within_0.50pp_of_matched_fly': protected['validation_average_accuracy'] >= fly['validation_average_accuracy'] - 0.50,
    'beats_standard_fd_by_0.10pp': protected['validation_average_accuracy'] >= standard['validation_average_accuracy'] + 0.10,
    'state_smaller_than_matched_fly': protected['persistent_state_bytes'] < fly['persistent_state_bytes'],
    'heldout_test_remained_locked': locked_test_path.is_file() and not test_path.exists(),
}
print(json.dumps(gates, indent=2))
print('PHASE A GATE:', 'PASS' if all(gates.values()) else 'FAIL — do not evaluate test')

In [ ]:
# Download train-only evidence. The held-out test tensor is deliberately excluded.
import shutil
artifact = shutil.make_archive('/content/pps_soho_phasea_train_only', 'zip', OUTPUT_DIR)
print('artifact:', artifact)
from google.colab import files
files.download(artifact)